# Lesson 2.3: RAG-Specific Prompting & Grounding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Prompt-Engineering-Mastery/blob/main/tutorial/04b-rag-prompting.ipynb)

In a production environment, you rarely ask an LLM to generate answers purely from its pre-trained weights. Instead, you build **Retrieval-Augmented Generation (RAG)** pipelines that query a database for relevant documents, paste those documents into the prompt, and ask the model to synthesize an answer.

However, LLMs are naturally conversational and eager to please. If the retrieved documents do not contain the answer, or if the documents contradict each other, standard prompts often lead the model to hallucinate or fall back on its pre-trained memory. 

This lesson teaches you how to write prompts that strictly ground the model in retrieved sources, resolve conflicts, and format precise citations.

---

**📍 Lesson Roadmap:**

| Section | Audience |
|:---|:---|
| 1. Grounding: Preventing Pre-Trained Drift | 🟢 Everyone |
| 2. Handling Source Conflicts | 🟢 Everyone |
| 3. Precise Citation Formatting | 🟢 Everyone |
| 4. Technical Implementation | 🔷 Technical — Python SDK RAG prompt execution |
| Concept Check | 🟢 Everyone |

---

## 🟢 1. Grounding: Preventing Pre-Trained Drift

**Grounding** is the practice of restricting the LLM's answer space to *only* the information provided in the prompt context. If a fact isn't in the provided documents, the model must say it doesn't know, rather than making up a plausible-sounding answer.

To ground a model effectively:
1. **Isolate the context** using clear XML delimiters (e.g., `<context>...</context>`).
2. **Provide a fallback instruction** that gives the model a safe exit (e.g., "If you cannot find the answer, state that you do not know").
3. **Negative constraint enforcement:** Explicitly forbid drawing on outside knowledge.

### Un-grounded Prompt (Bad)
```text
Answer the user query based on these notes:
[Notes text here]

Query: When was the system migrated to PostgreSQL?
```
*Risk:* If the notes don't mention PostgreSQL, the model might search its pre-trained weights for general PostgreSQL release dates or make a guess.

### Grounded Prompt (Good)
```text
You are a factual Q&A assistant. Answer the user query using ONLY the documents provided inside the <context> tags.

Rules:
1. Ground your answer strictly in the provided context. Do NOT extrapolate or use pre-trained knowledge.
2. If the context does not contain the answer to the query, respond exactly with: "I cannot answer this question based on the provided documents."

<context>
[Retrieved Document 1]
[Retrieved Document 2]
</context>

Query: When was the system migrated to PostgreSQL?
```

> **💡 Pro Tip:** Anthropic's Claude is exceptionally good at staying grounded when instructed with XML blocks. OpenAI's reasoning models (o1/o3) are also highly effective, but standard instructions are still required to define what the model should do if information is missing.

---

## 🟢 2. Handling Source Conflicts

In real-world RAG applications, retrieved documents can contain outdated information, duplicates, or outright contradictions (e.g., Document A says the project deadline is Friday, but Document B says it was extended to next Tuesday).

Without instructions on how to handle conflicts, the model will either choose one randomly or output a confusing response that states both as absolute truth.

### Strategies for Conflict Resolution in Prompts:

1. **Recency-based resolution:** Instruct the model to trust the most recent document if dates or timestamps are available.
2. **Authority-based hierarchy:** Instruct the model to prioritize specific source types (e.g., "Trust documentation from `official-docs` over `slack-chats`").
3. **Acknowledge and expose:** Direct the model to explicitly point out the conflict to the user.

### Example Prompt for Source Conflicts
```text
You are an analyst. You will be given documents from multiple sources. If the documents contradict each other, resolve the conflict using the following rules in order of priority:
1. Prioritize documents with a newer "last_updated" timestamp.
2. If timestamps are missing or equal, prioritize "Official Policies" over "Support Tickets".
3. If the conflict cannot be resolved, state the contradiction clearly (e.g., "Document A states X, but Document B states Y").

<documents>
  <doc id="1" type="Support Tickets" timestamp="2026-05-10">
    Refund policy is 30 days.
  </doc>
  <doc id="2" type="Official Policies" timestamp="2026-04-01">
    Refund policy is 14 days.
  </doc>
</documents>

Query: What is the refund policy?
Answer:
```
*Expected Response:* "According to the Support Ticket from 2026-05-10 (which is more recent), the refund policy is 30 days."

---

## 🟢 3. Precise Citation Formatting

A RAG system is only as trustworthy as its citations. If a user cannot verify *where* the model found a fact, they will not trust the system.

You must instruct the model to cite the exact document identifiers (IDs) next to every claim it makes.

### Citation Instructions Checklist
* **Cite inline:** Instruct the model to place citations directly after the sentence or clause containing the fact, not just as a list at the end.
* **Strict format:** Enforce a machine-readable citation format (e.g., `[Doc ID]` or `[Source Name, Page Number]`).
* **Verify citations:** Tell the model that every claim must be backed by a cited document, and uncited facts are forbidden.

### Example Prompt
```text
Answer the user's question using the documents below. 

For every statement or fact you write, you MUST add an inline citation referencing the document ID (e.g., [1] or [3]). Place the citation immediately after the sentence or clause it supports.

<context>
  <doc id="doc_9a">
    Acme Corp offers standard shipping (3-5 business days) and express shipping (1-2 business days).
  </doc>
  <doc id="doc_12b">
    Express shipping is free for orders over $150. Otherwise, it costs $15.
  </doc>
</context>

Query: How much does it cost to get express shipping for a $50 order?
Answer:
```
*Expected Response:* "Express shipping costs $15 for a $50 order [doc_12b], and takes 1-2 business days to arrive [doc_9a]."

---

## 🔷 4. Technical Implementation

Here is how to implement a grounded, citing RAG prompt in Python using the OpenAI SDK. We wrap the context in XML tags and enforce strict schema rules.

In [ ]:
from openai import OpenAI

client = OpenAI()

# Simulated retrieved documents from a Vector Database
retrieved_docs = [
    {"id": "doc_1", "content": "Our API rate limit is 100 requests per minute for the free tier."},
    {"id": "doc_2", "content": "Paid tier subscribers get 5,000 requests per minute and priority support."},
]

# Formatting the retrieved documents into XML strings
context_str = ""
for doc in retrieved_docs:
    context_str += f"<document id='{doc['id']}'>\n{doc['content']}\n</document>\n"

user_query = "What are the rate limits for free users?"

response = client.chat.completions.create(
    model="gpt-4o",
    temperature=0.0,  # Factual extraction requires 0.0 temperature
    messages=[
        {
            "role": "system",
            "content": (
                "You are a factual QA bot. Answer user questions using only the text inside the <context> tags.\n"
                "For every fact you state, append an inline citation pointing to the document ID (e.g., [doc_1]).\n"
                "If the context does not contain the answer, reply with: 'I do not know.'"
            )
        },
        {
            "role": "user",
            "content": f"<context>\n{context_str}</context>\n\nQuery: {user_query}"
        }
    ]
)

print(response.choices[0].message.content)
# Output: "Free users have an API rate limit of 100 requests per minute [doc_1]."

---

## 🟢 Concept Check

**Scenario:** You have built a customer support RAG bot. A user asks: "What is your return policy?" The retrieved context is empty because the database query failed. If you did not write grounding rules, what is the model most likely to do?

* [ ] **A)** Throw an API error and crash the application.
* [ ] **B)** Refuse to answer because it realizes there is no context.
* [x] **C)** Answer based on general, pre-trained knowledge of common corporate return policies (e.g., "Most companies offer a 30-day return policy..."), potentially hallucinating policies your company doesn't support.
* [ ] **D)** Ask the user to upload the return policy document themselves.

<details>
<summary><b>🔑 Click to Reveal Answer & Explanation</b></summary>

**Correct Answer: C**

**Explanation:** LLMs are trained to be helpful and conversational autocomplete engines. Without explicit rules telling them *not* to use their pre-trained weights, they will default to answering using whatever patterns they have in memory. This is a primary source of "hallucinations" in production RAG systems. Adding a strict constraint like "If the context is empty or missing the answer, say 'I don't know'" is crucial to preventing this behavior.
</details>

---

## 📚 References & Further Reading

* **Retrieval-Augmented Generation (Lewis et al., 2020):** The seminal paper introducing the concept of RAG — [arXiv:2005.11401](https://arxiv.org/abs/2005.11401).
* **Mitigating Hallucinations in RAG:** A comprehensive overview of prompting techniques for grounded generation — [arXiv:2401.05856](https://arxiv.org/abs/2401.05856).
* **OpenAI Prompt Engineering Guide (Citations):** Best practices for generating citations via system instructions — [platform.openai.com/docs/guides/prompt-engineering](https://platform.openai.com/docs/guides/prompt-engineering).

---

## 🚀 What's Next?

Now that you can ground your models in external knowledge and cite sources, it's time to learn how to keep your prompts secure from malicious users. In the next module, we'll learn about delimiters, system prompts, safety classifiers, and defensive design.

* [Lesson 3.1: Delimiters, System Prompts, and Guardrails →](./05-delimiters-guardrails.mdx)